In [45]:
import json

from openai import OpenAI
from scraper import fetch_website_contents, fetch_website_links
from IPython.display import Markdown, display, update_display

In [46]:
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [47]:
links = fetch_website_links("https://edwarddonner.com")

In [48]:
print(links)

['#wp--skip-link--target', 'https://edwarddonner.com/avatar/', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/proficient/', 'https://edwarddonner.com/connect-four/', 'https://edwarddonner.com/outsmart/', 'https://edwarddonner.com/about-me-and-about-nebula/', 'https://edwarddonner.com/posts/', 'https://edwarddonner.com/', 'https://news.ycombinator.com', 'https://nebula.io/?utm_source=ed&utm_medium=referral', 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/avatar/', 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/', 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/', 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/', 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/', 'https://edwarddonner.com/

In [49]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should responed in JSON as in this example:
{
    "links": [
        {
            "type": "about page",
            "url": "https://full.url/goes/here/about",
        },
        {
            "type": "careers page",
            "url": "https://another.full.url/careers",
        },
    ]
}
"""

In [50]:
def get_links_user_prompt(url: str) -> str:
    user_prompt: str = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company,
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.
    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt
    

In [51]:
print(get_links_user_prompt("https://edwarddonner.com"))


    Here is the list of links on the website https://edwarddonner.com -
    Please decide which of these are relevant web links for a brochure about the company,
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.
    #wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-

In [52]:
def select_relevant_links(url: str):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {
                "role": "system",
                "content": link_system_prompt
            },
            {
                "role": "user",
                "content": get_links_user_prompt(url)
            }
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content

    if result is not None:
        links = json.loads(result)
        return links

    return None

In [53]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'About page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'Careers/Jobs page', 'url': None},
  {'type': ' LinkedIn profile',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': ' Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': ' Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [57]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'homepage', 'url': 'https://edwarddonner.com'},
  {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [ ]:
def fetch_page_and_all_relevant_links(url: str):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page: {contents}\n##Relevant Links:\n"

    if relevant_links is not None:
        for link in relevant_links["links"]:
            result += f"\n\nLink: {link['type']}"
            result += fetch_website_contents(link["url"])

    return result

In [58]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

CONTENTS HERE: Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 900,000 enrollments across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m su